In [1]:
from contextlib import contextmanager
import datetime as dt
import json
import zipfile

import pandas as pd

import src

In [ ]:
CHANNELS = [
    "@AfDFraktionimBundestag",
    "@AfDTV",
    "@cducsu",
    "@cdutv",
    "@csuimbundestag9622",
    "@csumedia",
    "@DieGruenen",
    "@gruenebundestag",
    "@DIELINKE",
    "@dielinkebt",
    "@linksfraktion",
    "@FDP",
    "@fdpbt",
    "@spdde",
    "@spdbt",
]

# Channel Metadata

In [3]:
def load_channel_meta(channel):
    path = src.PATH / "data/raw/yt" / channel / "channel_metadata.json"

    with path.open() as f:
        d = json.load(f)

    info = {
        "channel_uploader_id": d["uploader_id"],
        "channel_follower_count": d["channel_follower_count"],
        "channel_collected": dt.datetime.fromtimestamp(d["epoch"]),
        "channel_description": d["description"],
        "channel": d["channel"],
        "channel_id": d["channel_id"],
        "channel_url": d["channel_url"],
    }

    return info


def create_channel_frame(channels):
    results = []
    for channel in channels:
        info = load_channel_meta(channel)
        results.append(info)

    df = pd.DataFrame(results)

    return df


channels = create_channel_frame(CHANNELS)
channels.head()

,channel_uploader_id,channel_follower_count,channel_collected,channel_description,channel,channel_id,channel_url
0,@AfDFraktionimBundestag,539000,2025-04-28 16:29:00,Offizieller Kanal der AfD-Fraktion im Deutsche...,AfD-Fraktion Bundestag,UC_dZp8bZipnjntBGLVHm6rw,https://www.youtube.com/channel/UC_dZp8bZipnjn...
1,@AfDTV,340000,2025-04-29 02:29:54,Offizieller YouTube-Kanal der Alternative für ...,AfD TV,UCq2rogaxLtQFrYG3X3KYNww,https://www.youtube.com/channel/UCq2rogaxLtQFr...
2,@cducsu,8050,2025-04-29 23:14:50,Wir nehmen dich mit vor und hinter die Kulisse...,CDU•CSU Fraktion,UCWpop4RlpejOFLebR0C1YaQ,https://www.youtube.com/channel/UCWpop4RlpejOF...
3,@cdutv,30800,2025-04-29 19:32:16,Die CDU ist die Volkspartei der Mitte. Seit 19...,CDU,UCKyWIEse3u7ExKfAWuDMVnw,https://www.youtube.com/channel/UCKyWIEse3u7Ex...
4,@csuimbundestag9622,4930,2025-04-29 18:49:40,Willkommen bei der CSU im Bundestag!\n\nNetiqu...,CSU im Bundestag,UCerYBm-F7LMpXOgFkSDJIQw,https://www.youtube.com/channel/UCerYBm-F7LMpX...


In [4]:
CHANNEL_FILE = src.PATH / "data/yt_metadata/channels.parquet"
CHANNEL_FILE.unlink(missing_ok=True)
channels.to_parquet(CHANNEL_FILE, compression="gzip")

# Video Metadata

In [21]:
@contextmanager
def load_video_meta_archive(channel):
    path = src.PATH / "data/raw/yt" / channel / "metadata.zip"
    archive = None
    try:
        archive = zipfile.ZipFile(path, "r")
        file_list = archive.infolist()
        yield archive, file_list
    finally:
        if archive is not None:
            archive.close()


def load_video_metadata(channel):
    all_videos = []
    with load_video_meta_archive(channel) as (archive, files):
        for file in files:
            with archive.open(file) as f:
                byte_content = f.read()
                content = byte_content.decode("utf-8")
                content = json.loads(content)
                # assert that file exists
                file_path = src.PATH / "data/raw/yt" / channel / "videos" / f"{content['id']}.m4a"
                try:
                    info = {
                        "video_id": content["id"],
                        "channel": channel,
                        "channel_id": content["channel_id"],
                        "video_title": content["title"],
                        "video_duration": content.get("duration"),
                        "video_views": content["view_count"],
                        "video_likes": content.get("like_count", None),
                        "video_comments": content["comment_count"],
                        "video_was_live": content["is_live"] | content["was_live"],
                        "video_description": content["description"],
                        "video_datetime_upload": content.get("timestamp"),
                    }
                except KeyError:
                    print(channel)
                    print(file)
                    raise

                if not info["video_datetime_upload"]:
                    # some videos might have no timestamp for no apparent reason ...
                    continue
                else:
                    info["video_datetime_upload"] = dt.datetime.fromtimestamp(
                        info["video_datetime_upload"],
                    )

                if not info["video_duration"]:
                    # for whatever reason, videos sometimes have not duration
                    continue

                all_videos.append(info)
                if not info["video_was_live"] and (
                    info["video_datetime_upload"] >= dt.datetime(2017, 12, 6)
                ):
                    assert file_path.is_file(), file_path

    return all_videos


def create_video_frame(channels):
    videos = []
    for channel in channels:
        channel_videos = load_video_metadata(channel)
        videos.extend(channel_videos)

    return pd.DataFrame(videos)

In [22]:
df = create_video_frame(CHANNELS)
df.head()

,video_id,channel,channel_id,video_title,video_duration,video_views,video_likes,video_comments,video_was_live,video_description,video_datetime_upload
0,w9VshhIL8ig,@AfDFraktionimBundestag,UC_dZp8bZipnjntBGLVHm6rw,Presseerklärung von Alice Weidel und Tino Chru...,511,277365,24327.0,2300.0,True,In der letzten Plenarwoche vor der Sommerpause...,2023-07-04 14:57:45
1,2f_Ww2xW0FQ,@AfDFraktionimBundestag,UC_dZp8bZipnjntBGLVHm6rw,BUNDESTAG LIVE - 55. Sitzung - AfD-Fraktion im...,28389,20268,595.0,23.0,True,09:00 Sitzungseröffnung \n09:00 Finanzielle ...,2022-09-23 17:24:54
2,AuuenVJ70g0,@AfDFraktionimBundestag,UC_dZp8bZipnjntBGLVHm6rw,Alexander Gauland zum Nato-Beitritt von Schwed...,251,150901,9647.0,747.0,False,Offizieller Kanal der AfD-Fraktion im Deutsche...,2022-07-08 18:00:06
3,sWe2F_3OjEE,@AfDFraktionimBundestag,UC_dZp8bZipnjntBGLVHm6rw,Anträge der AfD-Fraktion zu kriminellen Migran...,2847,6611,771.0,74.0,False,Folge uns auch auf Telegram: https://t.me/afdf...,2018-10-16 13:03:38
4,sat8i3WIR5M,@AfDFraktionimBundestag,UC_dZp8bZipnjntBGLVHm6rw,Stephan Brandner macht Klartextansage! Ampel-P...,275,116018,11564.0,834.0,False,Offizieller Kanal der AfD-Fraktion im Deutsche...,2023-01-27 17:00:35


In [7]:
VIDEO_FILE = src.PATH / "data/yt_metadata/videos.parquet"
VIDEO_FILE.unlink(missing_ok=True)
df.to_parquet(VIDEO_FILE, compression="gzip")